# Hybrid Experiments: Latest XGB + Fusion

Plan... & Notes

### Imports

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import xgboost as xgb
import numpy as np
import librosa as lb
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras import models, layers, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

## Branch 1: **Engineered XGBoost Features**

Dataset now features:

- 12 kHz target sample rate,
- 6-second chunks,
- overlap for non-COPD classes using a 2-second step,
- no overlap for COPD,
- Mu-law compression,
- and a richer feature set including stds, bandwidth, skewness/kurtosis, and MFCCs up to 15.

### Load XGB data

In [2]:
xgb_df = pd.read_csv("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Extracted features/xgboost_features_6s_12kHz_compressed_after_normalization_w_overlapping_mfcc15.csv")

xgb_df.head()

,rms_mean,rms_std,zcr_mean,centroid_mean,centroid_std,flatness_mean,flatness_std,rolloff_mean,flux_mean,bandwidth_mean,...,mfcc_13_mean,mfcc_13_std,mfcc_14_mean,mfcc_14_std,mfcc_15_mean,mfcc_15_std,patient_id,chunk_id,original_file,diagnosis
0,0.740132,0.083106,0.001316,98.066230,93.868556,0.000034,0.000146,119.431516,1.826715,340.148743,...,8.284259,2.947182,8.153688,3.566122,7.156815,3.025179,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696974,0.079933,0.001572,94.479942,81.417895,0.000041,0.000361,107.338763,1.494651,338.655438,...,8.452517,3.363525,7.557367,4.090311,6.353885,4.028513,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670958,0.079359,0.001590,90.514388,75.801911,0.000024,0.000205,99.817154,1.613105,339.537651,...,8.950355,2.893862,7.780566,3.040372,6.503709,3.147859,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.675636,0.093185,0.001153,81.297469,52.764923,0.000008,0.000023,81.740359,1.693879,322.028853,...,8.722843,2.706289,8.697692,3.737317,6.902027,3.432181,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.635533,0.069224,0.001202,80.768717,67.285689,0.000014,0.000083,81.865027,1.705645,325.092538,...,9.458111,3.485257,10.628664,4.948114,8.951859,4.647463,223,4,223_1b1_Pr_sc_Meditron.wav,COPD


In [3]:
xgb_df.shape, xgb_df.columns, xgb_df.dtypes

((3606, 46),
 Index(['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std',
        'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean',
        'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean',
        'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std',
        'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean',
        'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std',
        'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std',
        'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std',
        'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std',
        'mfcc_15_mean', 'mfcc_15_std', 'patient_id', 'chunk_id',
        'original_file', 'diagnosis'],
       dtype='object'),
 rms_mean          float64
 rms_std           float64
 zcr_mean          float64
 centroid_mean     float64
 centroid_std      float64
 flatness_mean     float64
 flatness_std      float64


### XGB preprocessing

#### Apply 5-class filter

In [4]:
# Filter to same 5 classes - COPD, pneumonia, healthy, URTI, bronchiectasis
classes_to_keep = ["COPD", "Pneumonia", "Healthy", "URTI", "Bronchiectasis"]

xgb_df_filtered = xgb_df[xgb_df["diagnosis"].isin(classes_to_keep)].copy()
xgb_df_filtered = xgb_df_filtered.reset_index(drop=True)

In [9]:
xgb_df_filtered["diagnosis"].value_counts()

diagnosis
COPD              2597
Pneumonia          296
Healthy            275
URTI               182
Bronchiectasis     128
Name: count, dtype: int64

#### Label-encode target

In [5]:
# Encode target
le = LabelEncoder()
xgb_df_filtered["target"] = le.fit_transform(xgb_df_filtered["diagnosis"])

# Create a dictionary mapping labels -> numbers
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

class_mapping

{'Bronchiectasis': 0, 'COPD': 1, 'Healthy': 2, 'Pneumonia': 3, 'URTI': 4}

### Train & Test with Grouped Cross Validation

In [6]:
# Define features and target
X = xgb_df_filtered.drop(columns=["original_file", "patient_id", "diagnosis", "target", "chunk_id"])
y = xgb_df_filtered["target"]

# Define groups (by patient)
groups = xgb_df_filtered["patient_id"]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of groups (patients): {groups.nunique()}")
print("\nConfirm feature columns:")
print(list(X.columns))


Feature matrix shape: (3478, 42)
Target shape: (3478,)
Number of groups (patients): 117

Confirm feature columns:
['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std', 'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean', 'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std', 'mfcc_15_mean', 'mfcc_15_std']


In [7]:
# Grouped Cross Validation (split by patient ID)
# IMPORTANT: We used StratifiedGroupKFold to preserve patient-level separation
# while improving class balance across folds.
sgkf = StratifiedGroupKFold(n_splits=3)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # Outer split: creates grouped train/test splits
    X_train_full = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_full = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train_full = groups.iloc[train_idx]

    # Inner split: creates grouped train/validation splits
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    idx_train, idx_val = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

    X_train = X_train_full.iloc[idx_train]
    X_val = X_train_full.iloc[idx_val]

    y_train = y_train_full.iloc[idx_train]
    y_val = y_train_full.iloc[idx_val]

    # Class-balance sample weights on TRAIN only
    w_train = compute_sample_weight(class_weight="balanced", y=y_train)

    # ==================
    # XGBoost model
    # ==================
    # UPDATED to match latest model from Antonella
    xgb_model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=3,
        subsample=0.8,        # only 80% of lines in each tree
        colsample_bytree=0.7, # only 70% of cols in each tree
        reg_lambda=1.5,
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        eval_metric='mlogloss',
        early_stopping_rounds=15
    )

    print("Train classes:", np.sort(y_train.unique()))
    print("Val classes:  ", np.sort(y_val.unique()))
    print("Test classes: ", np.sort(y_test.unique()))

    # Train
    xgb_model.fit(
        X_train,
        y_train,
        sample_weight=w_train, # gives more importance to rare classes
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Predict
    y_pred = xgb_model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Fold {fold} accuracy   : {acc:.4f}")
    print(f"Fold {fold} macro F1   : {macro_f1:.4f}")
    print(f"Fold {fold} weighted F1: {weighted_f1:.4f}\n")

    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_,
        zero_division=0
    ))

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

# =========================================================
# Summary across folds
# =========================================================
results_df = pd.DataFrame(fold_results)

print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY")
print("="*60)
print(results_df)

print("\nMean metrics:")
print(results_df[["accuracy", "macro_f1", "weighted_f1"]].mean())



FOLD 1
Train classes: [0 1 2 3 4]
Val classes:   [1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 1 accuracy   : 0.7185
Fold 1 macro F1   : 0.4369
Fold 1 weighted F1: 0.7377

                precision    recall  f1-score   support

Bronchiectasis       0.58      0.62      0.60        40
          COPD       0.94      0.84      0.89       865
       Healthy       0.27      0.51      0.35        94
     Pneumonia       0.25      0.27      0.26        96
          URTI       0.09      0.08      0.09        63

      accuracy                           0.72      1158
     macro avg       0.43      0.47      0.44      1158
  weighted avg       0.77      0.72      0.74      1158


FOLD 2
Train classes: [0 1 2 3 4]
Val classes:   [0 1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 2 accuracy   : 0.7961
Fold 2 macro F1   : 0.4786
Fold 2 weighted F1: 0.7832

                precision    recall  f1-score   support

Bronchiectasis       0.52      0.40      0.45        40
          COPD       0.90      0.95     

In [29]:
# Probabilities (not predictions!)
xgb_val_proba = xgb_model.predict_proba(X_val)
xgb_test_proba = xgb_model.predict_proba(X_test)

## Branch 2: **CNN**

Adapt config to latest XGBoost set-up

In [8]:
cnn_df = xgb_df_filtered.copy()

cnn_df.shape, cnn_df["diagnosis"].value_counts()

((3478, 47),
 diagnosis
 COPD              2597
 Pneumonia          296
 Healthy            275
 URTI               182
 Bronchiectasis     128
 Name: count, dtype: int64)

### Set-up with new chunk logic

In [13]:
# Match XGBoost preprocessing
# - UPDATED: 12 kHz sample rate
# - fixed 6-second slices
TARGET_SR = 12000
CHUNK_DURATION = 6
SAMPLES_PER_CHUNK = TARGET_SR * CHUNK_DURATION

RAW_AUDIO_FOLDER = Path("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files")

# NEW Overlap logic
STEP_COPD = SAMPLES_PER_CHUNK          # 6s step
STEP_NON_COPD = TARGET_SR * 2          # 2s step

In [14]:
def load_one_chunk_with_overlap(original_file, chunk_id, diagnosis, raw_audio_folder):
    """
    Reconstructs the exact chunk using Antonella's logic:
    - COPD → no overlap (6s step)
    - Others → 2s step (overlapping)
    """

    # Load audio at 12kHz
    y, _ = lb.load(raw_audio_folder / original_file, sr=TARGET_SR)

    # Trim silence (same as before)
    y, _ = lb.effects.trim(y)

    # Choose step size based on class
    if diagnosis == "COPD":
        step = STEP_COPD
    else:
        step = STEP_NON_COPD

    # Create chunks using correct step
    chunks = [
        y[i:i + SAMPLES_PER_CHUNK]
        for i in range(0, len(y), step)
        if len(y[i:i + SAMPLES_PER_CHUNK]) == SAMPLES_PER_CHUNK
    ]

    # Return correct chunk
    return chunks[int(chunk_id)].astype(np.float32)

### Create MelSpec from new chunks

In [15]:
def chunk_to_mel(chunk):
    mel = lb.feature.melspectrogram(
        y=chunk,
        sr=TARGET_SR,
        n_mels=128
    )

    mel_db = lb.power_to_db(mel, ref=np.max)

    # Add channel dimension for CNN
    return mel_db[..., np.newaxis].astype(np.float32)

In [16]:
def build_cnn_dataset(df_subset, raw_audio_folder):
    """
    Builds mel spectrogram dataset aligned with new chunking logic
    """

    X_list = []
    y_list = []

    for row in df_subset.itertuples(index=False):
        chunk = load_one_chunk_with_overlap(
            row.original_file,
            row.chunk_id,
            row.diagnosis,
            raw_audio_folder
        )

        mel = chunk_to_mel(chunk)

        X_list.append(mel)
        y_list.append(row.target)

    return np.stack(X_list), np.array(y_list)

### Reuse existing splits for CNN

In [18]:
# Grouped split (using same logics as XGBoost)
X_dummy = cnn_df.drop(columns=["diagnosis", "target"])
y = cnn_df["target"]
groups = cnn_df["patient_id"]

# NEW StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=3)
train_idx, test_idx = next(sgkf.split(X_dummy, y, groups=groups))

cnn_df_train_full = cnn_df.iloc[train_idx]
cnn_df_test = cnn_df.iloc[test_idx]

gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

idx_train, idx_val = next(
    gss.split(cnn_df_train_full, cnn_df_train_full["target"], groups=cnn_df_train_full["patient_id"])
)

cnn_df_train = cnn_df_train_full.iloc[idx_train]
cnn_df_val = cnn_df_train_full.iloc[idx_val]

In [19]:
# Build mel datasets
X_train_img, y_train = build_cnn_dataset(cnn_df_train, RAW_AUDIO_FOLDER)
X_val_img, y_val = build_cnn_dataset(cnn_df_val, RAW_AUDIO_FOLDER)
X_test_img, y_test = build_cnn_dataset(cnn_df_test, RAW_AUDIO_FOLDER)

In [20]:
print(X_train_img.shape)
print(y_train.shape)

(1608, 128, 141, 1)
(1608,)


In [21]:
# Checking class representation
print(np.unique(y_train))

[0 1 2 3 4]


### Train and evaluate CNN

In [24]:
# One-hot encode labels
num_classes = len(np.unique(y_train))

y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [25]:
def build_cnn(input_shape, num_classes):
    cnn_model = models.Sequential()

    #Input
    cnn_model.add(layers.Input(shape=input_shape))

    # Conv2D Block 1
    cnn_model.add(layers.Conv2D(32, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 2
    cnn_model.add(layers.Conv2D(64, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 3
    cnn_model.add(layers.Conv2D(128, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Conv2D Block 4
    cnn_model.add(layers.Conv2D(256, (3, 3), padding="same"))
    cnn_model.add(layers.BatchNormalization())
    cnn_model.add(layers.Activation("relu"))
    cnn_model.add(layers.MaxPooling2D((2, 2)))

    # Turn feature maps into one vector
    cnn_model.add(layers.GlobalMaxPooling2D())

    # Dense layer before classification
    cnn_model.add(layers.Dense(32, activation="relu"))
    cnn_model.add(layers.Dropout(0.3))

    # Final prediction layer
    cnn_model.add(layers.Dense(num_classes, activation="softmax"))

    cnn_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return cnn_model

cnn_model = build_cnn(X_train_img.shape[1:], num_classes)
cnn_model.summary()

2026-03-24 10:59:25.092978: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-24 10:59:25.093322: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-24 10:59:25.093335: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-24 10:59:25.093546: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-24 10:59:25.093566: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 141, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 141, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 141, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 70, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 70, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 70, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 70, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 35, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 35, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 35, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 35, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 17, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 17, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 17, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 398,149 (1.52 MB)

 Trainable params: 397,189 (1.52 MB)

 Non-trainable params: 960 (3.75 KB)

In [26]:
cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_data=(X_val_img, y_val_cat),
    epochs=30,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

Epoch 1/30


2026-03-24 11:00:08.380160: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


51/51 ━━━━━━━━━━━━━━━━━━━━ 9s 125ms/step - accuracy: 0.6150 - loss: 2.1808 - val_accuracy: 0.7837 - val_loss: 1.8643
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.6561 - loss: 1.5313 - val_accuracy: 0.7542 - val_loss: 0.7636
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.6965 - loss: 1.2995 - val_accuracy: 0.5070 - val_loss: 1.3314
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.7133 - loss: 1.4295 - val_accuracy: 0.7739 - val_loss: 0.6445
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - accuracy: 0.7506 - loss: 1.2337 - val_accuracy: 0.5183 - val_loss: 1.7295
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 99ms/step - accuracy: 0.7730 - loss: 1.1242 - val_accuracy: 0.7584 - val_loss: 0.8177
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.7873 - loss: 1.0049 - val_accuracy: 0.2570 - val_loss: 5.3080
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 100ms/step - accuracy: 0.8153 - loss: 0.8000 - val_accuracy: 0.7865 - val_loss: 

In [27]:
cnn_proba = cnn_model.predict(X_test_img)

cnn_proba

37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step


array([[0.0027228 , 0.03114727, 0.66147846, 0.23306406, 0.07158745],
       [0.00229934, 0.02507832, 0.648669  , 0.25688934, 0.06706399],
       [0.00359559, 0.04222836, 0.61694694, 0.22522192, 0.11200725],
       ...,
       [0.00254627, 0.13939436, 0.7700296 , 0.02391909, 0.06411058],
       [0.0012977 , 0.832922  , 0.14838111, 0.00499414, 0.0124051 ],
       [0.00265856, 0.771891  , 0.19247992, 0.01241792, 0.0205526 ]],
      dtype=float32)

In [28]:
cnn_proba.shape

(1158, 5)

In [30]:
cnn_val_proba = cnn_model.predict(X_val_img)
cnn_test_proba = cnn_model.predict(X_test_img)

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


## Fusion: XGB and CNN

In [33]:
print(xgb_val_proba.shape)
print(cnn_val_proba.shape)

(382, 5)
(712, 5)


In [32]:
# Find optimum weighting

weights = np.linspace(0, 1, 6)

for w in weights:
    final_proba = w * xgb_val_proba + (1 - w) * cnn_val_proba
    y_pred = np.argmax(final_proba, axis=1)

    print(w, f1_score(y_val, y_pred, average="macro"))

ValueError: operands could not be broadcast together with shapes (382,5) (712,5) 